In [14]:
import pandas as pd
from sqlalchemy import create_engine

In [15]:
# Load Combined Stock Data

df = pd.read_csv("C:/Users/arunm/Downloads/combined_stock_data.csv")

In [16]:
# Clean numeric columns 
num_cols = ['open','high','low','close','volume']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["Ticker","date"])


In [17]:
# Daily Return
df["Daily_Return"] = df.groupby("Ticker")["close"].pct_change()

In [18]:
# Cumulative returns
df["Cumulative_Return"] = (1 + df["Daily_Return"]).groupby(df["Ticker"]).cumprod()

In [19]:
# Yearly return
yearly_return = (
    df.groupby("Ticker")["Cumulative_Return"]
      .last()
      .reset_index()
      .rename(columns={"Cumulative_Return":"Yearly_Return"})
)

In [20]:
# Top gainers
top_green = yearly_return.sort_values("Yearly_Return", ascending=False).head(10)

# Top losers
top_red = yearly_return.sort_values("Yearly_Return", ascending=True).head(10)

In [21]:
# Volatility
vol = (
    df.groupby("Ticker")["Daily_Return"]
      .std()
      .reset_index()
      .rename(columns={"Daily_Return":"Volatility"})
)


In [22]:
# Cumulative return trend
cum = (
    df.groupby(["Ticker","date"])["Cumulative_Return"]
      .last()
      .reset_index()
)


In [23]:
# Correlation matrix
pivot = df.pivot(index="date", columns="Ticker", values="close")
corr = pivot.corr()

In [24]:
# PostgreSQL Connection


username = "postgres"
password = "kovilvenni"
host     = "localhost"
port     = "5432"
database = "stockdb"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

In [25]:
# Write all tables into PostgreSQL


df.to_sql("stock_prices", engine, if_exists="replace", index=False)

yearly_return.to_sql("yearly_returns", engine, if_exists="replace", index=False)

top_green.to_sql("top10_green", engine, if_exists="replace", index=False)

top_red.to_sql("top10_red", engine, if_exists="replace", index=False)

vol.to_sql("volatility", engine, if_exists="replace", index=False)

cum.to_sql("cumulative_returns", engine, if_exists="replace", index=False)



corr.to_sql("correlation_matrix", engine, if_exists="replace")

print("All data inserted into PostgreSQL successfully!")

All data inserted into PostgreSQL successfully!
